# Misconception #7: a tiny effect can have a tiny P value

**Keep the true effect fixed at just 0.001 standard deviations. Increase the sample size. What happens to P?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy.stats import norm
from IPython.display import Markdown, display

%matplotlib inline

plt.rcParams.update({
    "figure.dpi": 130,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
BLUE, GRAY, ORANGE = "#236A9F", "#7A828C", "#BF650F"


In [ ]:
DELTA = 0.01     # True mean shift in outcome units: 0.001 SD with SIGMA = 1.
SIGMA = 1.0       # Known population SD; fixed as N increases.
ALPHA = 0.05      # Reference significance threshold.
N_REPS = 10_000   # Independent studies at each N.
SEED = 987987
N_VALUES = np.unique(np.logspace(2, 8, 49).round().astype(int))

print(f"True standardized effect: {DELTA / SIGMA:g} SD")
print(f"{N_REPS:,} studies at each of {len(N_VALUES)} sample sizes")
print(f"N ranges from {N_VALUES[0]:,} to {N_VALUES[-1]:,} independent observations")


## The simplest model

Each study has **N independent observations** from a normal population:

$$
X_i \sim \mathcal{N}(\delta,\sigma^2), \qquad H_0:\delta=0.
$$

We use a two-sided, one-sample **z test with known population SD**. This deliberately keeps the calculation simple.

$$\mathrm{SE}(\bar X)=\frac{\sigma}{\sqrt{N}}, \qquad
Z=\frac{\bar X}{\sigma/\sqrt{N}}, \qquad
P=2\Phi(-|Z|).$$

The numerator stays near the same tiny true effect while the standard error shrinks with $1/\sqrt{N}$. A particular study's estimate still fluctuates.

## First, hold the observed estimate exactly fixed

Suppose every study happened to estimate $\bar X=\delta$. Then P is a deterministic function of N, given that estimate and the known SD:

$$P(N)=2\Phi\!\left(-\frac{|\delta|\sqrt{N}}{\sigma}\right).$$

The table holds the **observed estimate** fixed, not just the true effect. 


In [ ]:
critical = norm.isf(ALPHA / 2)
selected = np.linspace(0, len(N_VALUES) - 1, 7).astype(int)
reference_rows = [
    "| N | Fixed estimate (SD units) | Confidence interval (SD units) | P |",
    "| ---: | ---: | :---: | ---: |",
]
for n in N_VALUES[selected]:
    se = SIGMA / np.sqrt(n)
    p = 2 * norm.sf(abs(DELTA) / se)
    lo, hi = (DELTA + np.array([-1, 1]) * critical * se) / SIGMA
    reference_rows.append(
        f"| {n:,} | {DELTA / SIGMA:.6f} | [{lo:.6f}, {hi:.6f}] | {p:.3g} |"
    )
display(Markdown(f"**{100 * (1 - ALPHA):g}% confidence intervals**\n\n" + "\n".join(reference_rows)))


## P across effect size and sample size

Each row fixes the observed estimate at $\bar X = \delta$, as in the reference table. We vary its magnitude across rows and N across columns, keeping $\sigma$ fixed. This is a reference calculation, not an average over simulated studies.

The color scale is logarithmic, with all P values below $10^{-12}$ sharing one color. The white curve marks the significance threshold; the dashed line marks the current effect size. Positive and negative estimates of the same magnitude give the same two-sided P.

At $\delta = 0$, this fixed-estimate calculation gives P = 1. Actual null-study P values still vary across samples.


In [ ]:
from matplotlib.colors import LogNorm

n_heat = np.unique(np.geomspace(N_VALUES[0], N_VALUES[-1], 300).round().astype(int))
delta_sd_heat = np.linspace(0, max(5 * abs(DELTA) / SIGMA, 0.001), 251)
z_heat = delta_sd_heat[:, None] * np.sqrt(n_heat[None, :])
p_heat = 2 * norm.sf(z_heat)
p_color_min = 1e-12  # Display floor only; the contour uses the original P values.

fig_heat, ax_heat = plt.subplots(figsize=(10, 5.5), layout="constrained")
image_heat = ax_heat.pcolormesh(
    n_heat, delta_sd_heat, np.maximum(p_heat, p_color_min),
    norm=LogNorm(vmin=p_color_min, vmax=1), cmap="viridis_r",
    shading="nearest", rasterized=True,
)
contours_heat = ax_heat.contour(
    n_heat, delta_sd_heat, p_heat, levels=[ALPHA], colors="white", linewidths=1.7,
)
ax_heat.clabel(contours_heat, fmt={ALPHA: f"P = {ALPHA:g}"}, fontsize=10)
ax_heat.set(
    xscale="log", xlim=(n_heat[0], n_heat[-1]),
    ylim=(delta_sd_heat[0], delta_sd_heat[-1]),
    xlabel="N (independent observations)",
    ylabel=r"Fixed estimate magnitude $|\delta| / \sigma$ (SD units)",
    title=r"P across effect size and N, holding $\bar X = \delta$",
)
cbar_heat = fig_heat.colorbar(image_heat, ax=ax_heat, extend="min")
cbar_heat.set_label("Two-sided P value (log color scale)")
cbar_heat.set_ticks([1e-12, 1e-8, 1e-4, ALPHA, 1])
cbar_heat.set_ticklabels([r"$10^{-12}$", r"$10^{-8}$", r"$10^{-4}$", f"{ALPHA:g}", "1"])
plt.show()


## Now repeat the studies: Monte Carlo

At each N, simulate fresh, independent studies, once with the tiny true effect $\delta$ and once with **zero true effect**. 

In [ ]:
standard_errors = SIGMA / np.sqrt(N_VALUES)

def simulate_studies(true_effect, seed):
    rng = np.random.default_rng(seed)
    estimates = rng.normal(
        loc=true_effect,
        scale=standard_errors,
        size=(N_REPS, len(N_VALUES)),
    )
    z = estimates / standard_errors
    p_values = 2 * norm.sf(np.abs(z))
    return estimates, p_values

effect_estimates, effect_p = simulate_studies(DELTA, SEED)
null_estimates, null_p = simulate_studies(0.0, SEED + 2)

effect_q = np.quantile(effect_p, [0.05, 0.5, 0.95], axis=0)
null_q = np.quantile(null_p, [0.05, 0.5, 0.95], axis=0)
effect_rate = (effect_p < ALPHA).mean(axis=0)
null_rate = (null_p < ALPHA).mean(axis=0)

# Exact two-sided power provides an independent reference for the simulation.
mean_z = DELTA / standard_errors
exact_power = norm.sf(critical - mean_z) + norm.cdf(-critical - mean_z)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.2), layout="constrained")
null_min = null_p.min(axis=0)
null_max = null_p.max(axis=0)
ax.fill_between(N_VALUES, null_min, null_max, color=GRAY, alpha=0.12,
                label="Zero effect: simulated min–max P")
ax.fill_between(N_VALUES, effect_q[0], effect_q[2], color=BLUE, alpha=0.18,
                label="Tiny effect: middle 90%")
ax.plot(N_VALUES, effect_q[1], color=BLUE, lw=2.5,
        label="Tiny effect: median P")
ax.plot(N_VALUES, null_q[1], color=GRAY, lw=2, ls="--",
        label="Zero effect: median P")
ax.axhline(ALPHA, color=ORANGE, lw=1.4, ls=":", label=f"P = {ALPHA:g}")
ax.set(xlim=(N_VALUES[0], N_VALUES[-1]), ylim=(0, 1.02),
       title="P versus N across repeated studies (linear axes)",
       xlabel="N (independent observations)", ylabel="Two-sided P value")
ax.grid(axis="both", alpha=0.18)
ax.legend(loc="upper right", fontsize=9, frameon=False)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), layout="constrained")
fig.suptitle(
    f"A true effect of {DELTA / SIGMA:g} SD, at every sample size\n"
    f"{N_REPS:,} independent studies at each N",
    fontsize=15,
)

ax = axes[0]
# A display floor only matters if parameters are changed to more extreme effects.
# It is never used to determine whether a study is significant.
plot_floor = 1e-300
ax.fill_between(N_VALUES, np.maximum(null_q[0], plot_floor),
                np.maximum(null_q[2], plot_floor), color=GRAY, alpha=0.12)
ax.fill_between(N_VALUES, np.maximum(effect_q[0], plot_floor),
                np.maximum(effect_q[2], plot_floor), color=BLUE, alpha=0.22,
                label="Tiny effect: middle 90%")
ax.plot(N_VALUES, np.maximum(effect_q[1], plot_floor), color=BLUE, lw=2.5,
        label="Tiny effect: median P")
ax.plot(N_VALUES, np.maximum(null_q[1], plot_floor), color=GRAY, lw=2, ls="--",
        label="Zero effect: median P")
ax.axhline(ALPHA, color=ORANGE, lw=1.4, ls=":", label=f"P = {ALPHA:g}")
ax.set(xscale="log", yscale="log", title="P versus N across repeated studies",
       xlabel="N (independent observations)", ylabel="Two-sided P value")
ax.set_ylim(max(effect_q[0].min() * 0.3, plot_floor), 1.1)
ax.legend(loc="lower left", fontsize=9, frameon=False)

ax = axes[1]
ax.plot(N_VALUES, exact_power, color=BLUE, lw=2.5, label="Tiny effect: exact power")
ax.plot(N_VALUES, effect_rate, "o", color=BLUE, ms=4,
        label="Tiny effect: Monte Carlo")
ax.plot(N_VALUES, null_rate, "o", color=GRAY, ms=4, mfc="white",
        label="Zero effect: Monte Carlo")
ax.axhline(ALPHA, color=GRAY, ls="--", lw=1.5,
           label=f"Zero effect: expected {100 * ALPHA:g}%")
ax.set(xscale="log", ylim=(-0.02, 1.03), title=f"How often is P < {ALPHA:g}?",
       xlabel="N (independent observations)", ylabel="Fraction of studies")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(loc="upper left", fontsize=9, frameon=False)

for ax in axes:
    ax.grid(axis="both", which="major", alpha=0.15)
plt.show()


In [ ]:
median_estimate = np.median(effect_estimates, axis=0) / SIGMA
mc_rows = [
    f"| N | Median estimated effect (SD) | Median P | Tiny effect: % P < {ALPHA:g} | Zero effect: % P < {ALPHA:g} |",
    "| ---: | ---: | ---: | ---: | ---: |",
]
for j in selected:
    mc_rows.append(
        f"| {N_VALUES[j]:,} | {median_estimate[j]:.6f} | {effect_q[1, j]:.3g} "
        f"| {100 * effect_rate[j]:.1f}% | {100 * null_rate[j]:.1f}% |"
    )
display(Markdown("\n".join(mc_rows)))


## The same P-versus-N plot on linear axes

This repeats the P-versus-N view above with **both axes linear**. The simulation and summary statistics are unchanged; only the display scale differs. For the zero-effect studies, the gray band shows the **minimum to maximum P value actually simulated** at each N, making their broad spread visible. Under the null, P values are uniform from 0 to 1, so this range is expected to span much of the plot. On a linear P axis, very small P values bunch near zero, while the early decline is easier to read in ordinary P-value units. The full N range is retained, so the largest sample sizes occupy most of the horizontal distance.
